<div style="background:linear-gradient(135deg,#1f4e79,#2f80ed 55%,#56ccf2);color:white;padding:26px 30px;border-radius:14px;margin:6px 0">
    <div style="font-size:2em;font-weight:800">
        Python Tools for Atmospheric Science
    </div>
    <div style="font-size:1.1em;opacity:.95;margin-top:6px">
        A demo tour: Xarray, GeoPandas, Cartopy, MetPy, Matplotlib, Plotly, and a few specialist packages
    </div>
</div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        ✍️ How to use this notebook...
    </div>
    <ul>
        <li>
            <b>Shift + Enter</b> runs a cell. <b>Do not use “Run All.”</b> Run the cells in order, because later cells use names made by earlier ones.
        </li>
        <li>
            Everything runs offline from the small files in <code>data/session_3</code>. The three cells in Section 8 that can fetch live data are marked <b>OPTIONAL</b> and stay offline unless you set <code>USE_LIVE = True</code>.
        </li>
        <li>
            Some packages are not installed by default. The cell in Section 1 shows the one-line install, commented out.
        </li>
        <li>
            This is a tour, not a course. Each tool gets one short passage and one cell you can change and run again.
        </li>
    </ul>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        🎯 Outline of this Course Module
    </div>
    <ul>
        <li>
            <b> Section 1: Which package opens which file? </b>
        </li>
        <li>
            <b> Section 2: Xarray, gridded data with labels </b>
        </li>
        <li>
            <b> Section 3: GeoPandas, shapes and where things are </b>
        </li>
        <li>
            <b> Section 4: Cartopy, maps </b>
        </li>
        <li>
            <b> Section 5: MetPy, units and soundings </b>
        </li>
        <li>
            <b> Section 6: Matplotlib, a figure you could hand to a colleague </b>
        </li>
        <li>
            <b> Section 7: Plotly, plots you can point at </b>
        </li>
        <li>
            <b> Section 8: Specialist tools (optional): ACT, PyRSIG, atmoz </b>
        </li>
    </ul>
</div>

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 1</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Which package opens which file?</span>
    <div style="opacity:.92">
        The data folder, and the tool each file points to
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Someone hands you a file called <code>geos_cf_midatlantic_aug2025.nc</code>. Which package opens it? Think about what it has to hold: PM2.5 on a grid of latitude and longitude, for every hour. That is three directions at once, and a table has only two. So pandas is the wrong tool, and the ending <code>.nc</code>, which stands for <b>NetCDF</b>, points to xarray.
</div>

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

# Look for the data folder in the usual places, in this order.
candidates = [Path('../data/session_3'), Path('data/session_3'), Path('/content/session_3'), Path('/content')]
for folder in candidates:
    if (folder / 'states_midatlantic.geojson').exists():
        DATA = folder
        break
else:
    raise FileNotFoundError(
        'Cannot find the Session 3 data files (states_midatlantic.geojson and the others). '
        'I looked in: ' + ', '.join(str(c) for c in candidates) + '. '
        'On Google Colab, upload everything from data/session_3 to /content/session_3, then run this cell again.'
    )

print('Using data folder:', DATA.resolve())

In [ ]:
# OPTIONAL. Needed only if metpy, act-atmos, pyrsig or atmoz are missing (Google Colab, for example).
# Remove the leading # on the lines below and run the cell once. If Colab asks you to restart the session, do so.
# %pip install metpy act-atmos pyrsig
# %pip install git+https://github.com/moroots/atmoz

In [ ]:
files = pd.DataFrame(
    [
        ('geos_cf_midatlantic_aug2025.nc', 'Model grid: PM2.5, NO2, O3 by hour, latitude, longitude', 'xarray', 2),
        ('states_midatlantic.geojson', 'State outlines (polygons)', 'GeoPandas', 3),
        ('pm25_daily_monitors_2025.parquet', 'Daily PM2.5 at 151 EPA monitors (a table)', 'pandas', 3),
        ('sounding_KIAD_2025-08-06_12Z.csv', 'Weather-balloon profile: pressure, temperature, dewpoint, wind', 'pandas, then MetPy', 5),
        ('sgpmetE13.b1.20190101.000000.cdf', 'ARM surface meteorology, one day', 'ACT', 8),
        ('asos_bwi_iad_dca_2025-08-05_06.parquet', 'Airport weather observations (a table)', 'ACT fetches it, pandas reads it', 8),
        ('tempo_no2_2026-10-20_midatlantic.parquet', 'Satellite NO2, one row per pixel', 'PyRSIG fetches it, pandas reads it', 8),
        ('tolnet_smol2_hartmiller_20250728_31.parquet', 'Lidar ozone profiles, time by altitude', 'atmoz', 8),
    ],
    columns=['file', 'what is in it', 'opened with', 'section'],
)
files['size (KB)'] = [round((DATA / name).stat().st_size / 1024) for name in files['file']]
files

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 2</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Xarray</span>
    <div style="opacity:.92">
        Gridded data that keeps its labels
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What is inside a NetCDF file? Open ours and print it. The file names three <b>dimensions</b>, <code>time</code>, <code>lat</code> and <code>lon</code>, and lists the values along each one, the <b>coordinates</b>: 78 hours, 37 latitudes, 53 longitudes. Each of the three <b>variables</b>, <code>pm25</code>, <code>no2</code> and <code>o3</code>, is a block of numbers laid out along all three, and each carries its own <b>attributes</b>, such as units. Together they make an xarray <b>Dataset</b>.
</div>

In [ ]:
ds = xr.open_dataset(DATA / 'geos_cf_midatlantic_aug2025.nc')
ds

In [ ]:
print(ds['pm25'].attrs)
print(ds['no2'].attrs)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do you ask for one day at one place without counting rows? By name. I will take the 24 hours that start at 05:00 UTC on 6 August, which is midnight Eastern standard time, and the grid cell nearest Baltimore. <code>.sel</code> takes a label rather than a position, and <code>method='nearest'</code> forgives a label that is not exactly on the grid. Then <code>.mean('time')</code> collapses the hour axis, and what is left is a map.
</div>

In [ ]:
local_day = slice('2025-08-06T05:00', '2025-08-07T04:59')

baltimore = ds.sel(lat=39.29, lon=-76.61, method='nearest')   # one grid cell, every hour
day_mean = ds.sel(time=local_day).mean('time')                 # every grid cell, one averaged day

print('Nearest grid cell to Baltimore (lat, lon):', float(baltimore.lat), float(baltimore.lon))
print('Hours in the day:', ds.sel(time=local_day).sizes['time'])
print('Baltimore cell, daily mean PM2.5:', round(float(day_mean['pm25'].sel(lat=39.29, lon=-76.61, method='nearest')), 1), 'ug/m3')

day_mean['pm25'].plot(cmap='YlOrRd')    # xarray's own quick plot: labels and units come from the file
plt.show()

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 3</span><span style="font-size:1.2em;font-weight:600"> &nbsp;GeoPandas</span>
    <div style="opacity:.92">
        Shapes, coordinate systems and where things are
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What does a state look like to a computer? Read the outlines and look. It is a table with one row per state, like any pandas table, plus a <code>geometry</code> column that holds the shape. That makes it a <b>GeoDataFrame</b>. It also remembers its <b>coordinate reference system</b>, here EPSG:4326, which means the numbers are degrees. Ask for an area in that system and you get square degrees, so I convert to an equal-area system, EPSG:5070, first.
</div>

In [ ]:
states = gpd.read_file(DATA / 'states_midatlantic.geojson')
states.head(3)

In [ ]:
print('CRS:', states.crs)
print('First outline, area in its own units (square degrees):', round(states.geometry.iloc[0].area, 2))

states['area_km2'] = (states.to_crs('EPSG:5070').area / 1e6).round()
states[['state', 'area_km2']].sort_values('area_km2').reset_index(drop=True)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Does every monitor sit in the state its record claims? Turn each site's latitude and longitude into a point and ask GeoPandas which polygon holds it. That is a <b>spatial join</b>. Every site that lands in a polygon lands in its own state, but three land in none. The outlines come from Natural Earth, drawn for maps at 1:10 million, so a site near a river or a bay can fall just outside.
</div>

In [ ]:
monitors = pd.read_parquet(DATA / 'pm25_daily_monitors_2025.parquet')
sites = monitors.drop_duplicates('site_id')

site_points = gpd.GeoDataFrame(sites, geometry=gpd.points_from_xy(sites.lon, sites.lat), crs='EPSG:4326')
polygons = states[['state', 'geometry']].rename(columns={'state': 'polygon_state'})
joined = gpd.sjoin(site_points, polygons, predicate='within', how='left')

inside = joined['polygon_state'].notna()
print(len(joined), 'sites;', inside.sum(), 'fall inside a polygon')
print('Inside a polygon, but in a different state from the table:', (inside & (joined['state'] != joined['polygon_state'])).sum())

joined[~inside][['site_name', 'state', 'county', 'lat', 'lon']]

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 4</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Cartopy</span>
    <div style="opacity:.92">
        Projections, transforms and maps
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How does a point on a round earth land on a flat page? Take Baltimore, 76.6 degrees west and 39.3 north. Those numbers are angles, and a map has to decide how to flatten them. That decision is the <b>projection</b>. I will use Lambert conformal, in which the lines of longitude lean toward each other as they run north. Then <code>transform=ccrs.PlateCarree()</code> tells Cartopy that the numbers I hand it are plain degrees, so it knows what to flatten.
</div>

In [ ]:
import cartopy.crs as ccrs

degrees = ccrs.PlateCarree()                                            # "these numbers are plain degrees"
projection = ccrs.LambertConformal(central_longitude=-77, central_latitude=39)

fig = plt.figure(figsize=(7, 6))
ax = plt.axes(projection=projection)
ax.set_extent([-84, -73, 34.5, 42.5], crs=degrees)

ax.add_geometries(states.geometry, crs=degrees, facecolor='#f2f2f2', edgecolor='0.3', linewidth=0.6)
ax.gridlines(linewidth=0.4, color='0.5')                                 # lines of latitude and longitude

ax.plot(-76.61, 39.29, 'o', color='tab:red', transform=degrees)          # transform= goes on every call that takes coordinates
ax.text(-76.4, 39.45, 'Baltimore', transform=degrees)
plt.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Why draw the model and the monitors on one color scale? A dot that matches the shading behind it blends in, and a dot that does not stands out. The shading is the GEOS-CF daily mean PM2.5 for 6 August, drawn with <code>pcolormesh</code>. The dots are the monitors' daily means, drawn with <code>scatter</code> and the same <code>cmap</code>, <code>vmin</code> and <code>vmax</code>. Over Pennsylvania the dots stand out against darker shading, and the means printed under the map agree: the model is about twice the monitors.
</div>

In [ ]:
m6 = monitors[monitors['date'] == '2025-08-06']
vmin, vmax = 0, 60

fig = plt.figure(figsize=(8, 6.5))
ax = plt.axes(projection=projection)
ax.set_extent([-84, -73, 34.5, 42.5], crs=degrees)

grid = ax.pcolormesh(day_mean.lon, day_mean.lat, day_mean['pm25'], transform=degrees,
                     cmap='YlOrRd', vmin=vmin, vmax=vmax, shading='nearest')
ax.add_geometries(states.geometry, crs=degrees, facecolor='none', edgecolor='0.25', linewidth=0.6)
ax.scatter(m6.lon, m6.lat, c=m6.pm25, s=45, cmap='YlOrRd', vmin=vmin, vmax=vmax,
           edgecolor='black', linewidth=0.6, transform=degrees)

fig.colorbar(grid, ax=ax, shrink=0.8, extend='max', label='PM2.5 (µg/m³), daily mean, 6 Aug 2025')
ax.set_title('GEOS-CF model (shading) and EPA monitors (dots)')

# pick the model value nearest each monitor, one value per monitor
nearest_cells = day_mean['pm25'].sel(lat=xr.DataArray(m6.lat.values), lon=xr.DataArray(m6.lon.values), method='nearest')
print(f'Mean over {len(m6)} monitors: monitors {m6.pm25.mean():.1f}, model {float(nearest_cells.mean()):.1f} ug/m3')
plt.show()

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 5</span><span style="font-size:1.2em;font-weight:600"> &nbsp;MetPy</span>
    <div style="opacity:.92">
        Units that travel with the numbers, and a sounding
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Why would you want Python to know that <code>30</code> is degrees Celsius? Multiply the number by a unit, <code>30 * units.degC</code>, and the meaning travels with it. Ask MetPy for the dewpoint of 30 &deg;C air at 60 percent relative humidity and the answer comes back labeled, ready to convert with <code>.to('degF')</code> or <code>.to('kelvin')</code>. I never type a conversion formula, so I cannot mistype one.
</div>

In [ ]:
import metpy.calc as mpcalc
from metpy.units import units

T = 30 * units.degC
RH = 60 * units.percent
Td = mpcalc.dewpoint_from_relative_humidity(T, RH)

print(f'Dewpoint: {Td:.1f~P}')
print(f'In Fahrenheit: {Td.to("degF"):.1f~P}')
print(f'Air temperature in kelvin: {T.to("kelvin"):.2f~P}')

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What can a weather balloon tell you that a surface thermometer cannot? Take the 12 UTC launch from Sterling, Virginia, on 6 August: temperature (red) and dewpoint (green) at 182 levels, from the ground to 5.5 hPa. At the surface the two are close, so the air is nearly saturated. Above about 930 hPa the dewpoint drops away and the air is dry. The dashed line follows a parcel lifted from the ground. On this <b>Skew-T</b> the temperature lines lean right, which keeps the curves from crowding.
</div>

In [ ]:
from metpy.plots import SkewT

snd = pd.read_csv(DATA / 'sounding_KIAD_2025-08-06_12Z.csv')
p = snd['pres'].values * units.hPa
t = snd['tmpc'].values * units.degC
td = snd['dwpc'].values * units.degC

parcel = mpcalc.parcel_profile(p, t[0], td[0]).to('degC')     # temperature of air lifted from the ground
lcl_p, lcl_t = mpcalc.lcl(p[0], t[0], td[0])                  # where that air would start to form cloud
theta = mpcalc.potential_temperature(p, t)                    # a derived quantity: temperature if brought down to 1000 hPa

# wind barbs at the standard pressure levels that have a wind reported
winds = snd[snd['pres'].isin([1000, 925, 850, 700, 500, 400, 300, 250, 200]) & snd['sknt'].notna()]
u, v = mpcalc.wind_components(winds['sknt'].values * units.knot, winds['drct'].values * units.degree)

fig = plt.figure(figsize=(7, 7))
skew = SkewT(fig, rotation=45)
fig.subplots_adjust(right=0.88)                              # leave room for the wind barbs
skew.plot_dry_adiabats(alpha=0.25)
skew.plot_moist_adiabats(alpha=0.25)
skew.plot_mixing_lines(alpha=0.25)
skew.plot(p, t, 'tab:red', label='Temperature')
skew.plot(p, td, 'tab:green', label='Dewpoint')
skew.plot(p, parcel, 'black', linestyle='--', label='Lifted parcel')
skew.plot_barbs(winds['pres'].values * units.hPa, u, v, length=6, xloc=1.05)   # drawn just outside the right edge
skew.ax.plot(lcl_t, lcl_p, marker='_', color='black', markersize=24)
skew.ax.set_ylim(1020, 200)
skew.ax.set_xlim(-30, 40)
skew.ax.set_xlabel('Temperature (°C)')
skew.ax.set_ylabel('Pressure (hPa)')
skew.ax.legend(loc='upper left')
skew.ax.set_title('Sterling, VA (KIAD), 6 Aug 2025, 12 UTC')
plt.show()

print(f'Lifted condensation level: {lcl_p:.0f~P}')
print(f'Potential temperature at the surface: {theta[0]:.1f~P}   at 850 hPa: {theta[snd["pres"] == 850][0]:.1f~P}')

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 6</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Matplotlib</span>
    <div style="opacity:.92">
        A figure you could hand to a colleague
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What separates a quick plot from one you could hand to a colleague? Take the PM2.5 data from Section 2 and make four decisions. <code>subplot_mosaic</code> puts one large map beside two time series. The color scale starts at zero and carries <code>extend='max'</code>, because the model goes past 60. Every axis says what it shows, in what units. An annotation marks the peak. Then <code>savefig</code> writes the file at 200 dots per inch.
</div>

In [ ]:
import matplotlib.dates as mdates

style = {'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False,
         'axes.titleweight': 'bold', 'axes.titlelocation': 'left'}

with plt.rc_context(style):
    fig, axs = plt.subplot_mosaic([['map', 'pm'], ['map', 'o3']], figsize=(11, 6),
                                  layout='constrained', width_ratios=[1.15, 1])

    # left: the daily-mean map, state outlines from GeoPandas on top
    ax = axs['map']
    shading = ax.pcolormesh(day_mean.lon, day_mean.lat, day_mean['pm25'],
                            cmap='YlOrRd', vmin=0, vmax=60, shading='nearest')
    states.boundary.plot(ax=ax, color='0.25', linewidth=0.6)
    ax.set_xlim(-84, -73)
    ax.set_ylim(34.5, 42.5)
    ax.set_aspect(1 / np.cos(np.radians(38.5)))          # keep distances roughly true
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    ax.set_title('Daily mean PM2.5, 6 Aug 2025')
    fig.colorbar(shading, ax=ax, shrink=0.85, extend='max', label='PM2.5 (µg m⁻³)')

    # right: hourly series for the grid cell nearest Baltimore
    ax = axs['pm']
    ax.plot(baltimore.time, baltimore['pm25'], color='tab:red', linewidth=2)
    ax.set_ylabel('PM2.5 (µg m⁻³)')
    ax.set_title('Grid cell nearest Baltimore, hourly')
    peak_time = baltimore['pm25'].idxmax('time')
    peak = float(baltimore['pm25'].max())
    ax.annotate(f'peak {peak:.0f}', xy=(peak_time.values, peak), xytext=(12, -4),
                textcoords='offset points', arrowprops={'arrowstyle': '->'})

    ax = axs['o3']
    ax.plot(baltimore.time, baltimore['o3'], color='tab:blue', linewidth=2)
    ax.set_ylabel('Ozone (ppb)')
    ax.set_xlabel('Time (UTC)')

    axs['pm'].sharex(axs['o3'])
    axs['pm'].tick_params(labelbottom=False)
    axs['o3'].xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 12]))
    axs['o3'].xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M'))

    fig.savefig('geos_cf_summary.png', dpi=200)
    print('Saved geos_cf_summary.png in', Path('.').resolve())
    plt.show()

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 7</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Plotly</span>
    <div style="opacity:.92">
        Plots that answer when you point at them
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    When a reader points at a dot and asks which monitor it is, where should the answer live? In the plot. Plotly draws the same monitors from the same table, with <code>hover_name</code> set to the site name, so the name, state and value appear when the mouse rests on a dot. The state outlines are not a map layer here. They are one more line trace, built from the GeoPandas polygons.
</div>

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# turn each state polygon into x and y lists, with None between shapes so the lines break there
outline_x, outline_y = [], []
for shape in states.geometry:
    for part in getattr(shape, 'geoms', [shape]):
        x, y = part.exterior.coords.xy
        outline_x += list(x) + [None]
        outline_y += list(y) + [None]

fig = px.scatter(
    m6, x='lon', y='lat', color='pm25', hover_name='site_name',
    hover_data={'state': True, 'pm25': ':.1f', 'lon': False, 'lat': False},
    color_continuous_scale='YlOrRd', range_color=[0, 40],
    labels={'pm25': 'PM2.5 (µg/m³)'}, title='EPA PM2.5 monitors, daily mean, 6 Aug 2025',
    width=760, height=560,
)
fig.update_traces(marker={'size': 10, 'line': {'width': 0.6, 'color': 'black'}})
fig.add_trace(go.Scatter(x=outline_x, y=outline_y, mode='lines', line={'color': 'gray', 'width': 1},
                         hoverinfo='skip', showlegend=False))
fig.update_xaxes(range=[-84, -73], title='Longitude (°E)')
fig.update_yaxes(range=[34.5, 42.5], title='Latitude (°N)',
                 scaleanchor='x', scaleratio=1 / np.cos(np.radians(38.5)))
fig.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Which state had the highest daily average of 2025, and when? You could sort a table. Or you could draw all eight lines and hover. Group the monitors by state and day, give <code>px.line</code> a date axis, and switch on the range slider. Hover over the tallest spike, Delaware on 14 June, and the box reads 43.5, with every other state listed beneath it for the same day.
</div>

In [ ]:
state_daily = monitors.groupby(['state', 'date'], as_index=False)['pm25'].mean()

fig = px.line(state_daily, x='date', y='pm25', color='state',
              labels={'pm25': 'PM2.5 (µg/m³)', 'date': 'Date'},
              title='Daily mean PM2.5 by state, 2025')
fig.update_traces(hovertemplate='%{y:.1f}')
fig.update_xaxes(rangeslider_visible=True)
fig.update_layout(hovermode='x unified', width=900, height=480)
fig.show()

peak = state_daily.loc[state_daily['pm25'].idxmax()]
print(f"Highest state daily mean: {peak['state']}, {peak['date']:%d %b %Y}, {peak['pm25']:.1f} ug/m3")

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 8</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Specialist tools (optional)</span>
    <div style="opacity:.92">
        ACT, PyRSIG and atmoz: each needs a pip install, and each has an offline fallback
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What would it take to plot a day of ARM data with its title, its units and the hours of daylight already on it? ACT's <code>read_arm_netcdf</code> opens an ARM file and keeps what the file says about itself. <code>TimeSeriesDisplay</code> then uses it: the title comes from the datastream name, the axis units come from the file, and one more call shades the daylight, worked out from the latitude and longitude stored in the file.
</div>

In [ ]:
import act    # the package is installed as act-atmos

arm = act.io.read_arm_netcdf(str(DATA / 'sgpmetE13.b1.20190101.000000.cdf'))   # ACT wants a text path, not a Path object
print(arm.attrs['location_description'])

disp = act.plotting.TimeSeriesDisplay(arm, subplot_shape=(2,), figsize=(9, 6))
disp.plot('temp_mean', subplot_index=(0,))
disp.day_night_background(subplot_index=(0,))
disp.plot('rh_mean', subplot_index=(1,))
plt.show()

In [ ]:
# OPTIONAL live fetch: hourly airport weather (ASOS) for BWI, IAD and DCA through ACT.
# USE_LIVE = False keeps the notebook offline and reads the bundled copy of the same request.
USE_LIVE = False

asos = None
if USE_LIVE:
    try:
        import datetime
        window = [datetime.datetime(2025, 8, 5), datetime.datetime(2025, 8, 7)]
        by_station = act.discovery.get_asos_data(window, station=['KBWI', 'KIAD', 'KDCA'])
        asos = pd.concat([d.to_dataframe().reset_index().assign(station=name) for name, d in by_station.items()])
        print('Fetched live.')
    except Exception as err:
        print('Live fetch failed (' + type(err).__name__ + '). Falling back to the bundled file.')

if asos is None:
    asos = pd.read_parquet(DATA / 'asos_bwi_iad_dca_2025-08-05_06.parquet')
    print('Using the bundled file.')

asos.groupby('station')['tmpf'].agg(['count', 'mean', 'max']).round(1)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do you get satellite NO2 for Maryland without downloading whole granule files and cutting them down yourself? Ask for a box and a time window. <code>RsigApi</code> sends both to the EPA's RSIG server, which returns only the pixels inside the box, one row per pixel, as a DataFrame. Ours is one hour of TEMPO data over the Mid-Atlantic, the 16 UTC hour on 20 October 2026.
</div>

In [ ]:
# OPTIONAL live fetch: TEMPO NO2 pixels from EPA's RSIG server through PyRSIG.
# USE_LIVE = False keeps the notebook offline and reads the bundled copy of the same request.
USE_LIVE = False

tempo = None
if USE_LIVE:
    try:
        import tempfile
        from pyrsig import RsigApi
        api = RsigApi(bdate='2026-10-20T16:00:00', edate='2026-10-20T16:59:59',
                      bbox=(-80, 36, -74, 40.5), workdir=tempfile.mkdtemp())   # bbox is west, south, east, north
        tempo = api.to_dataframe('TEMPO2.l2.no2.vertical_column_troposphere', backend='ascii', unit_keys=False)
        print('Fetched live.')
    except Exception as err:
        print('Live fetch failed (' + type(err).__name__ + '). Falling back to the bundled file.')

if tempo is None or tempo.empty:
    tempo = pd.read_parquet(DATA / 'tempo_no2_2026-10-20_midatlantic.parquet')
    print('Using the bundled file.')

print(len(tempo), 'pixels')
tempo.head(3)

In [ ]:
no2 = tempo['no2_vertical_column_troposphere'] / 1e15

fig, ax = plt.subplots(figsize=(8, 5.5))
pixels = ax.scatter(tempo['LONGITUDE'], tempo['LATITUDE'], c=no2, s=8, marker='s', linewidths=0,
                    cmap='viridis', vmin=0, vmax=no2.quantile(0.99))
states.boundary.plot(ax=ax, color='black', linewidth=0.7)
ax.set_xlim(-80, -74)
ax.set_ylim(36, 40.5)
ax.set_aspect(1 / np.cos(np.radians(38.2)))
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('TEMPO tropospheric NO2, 20 Oct 2026, 16 UTC hour')
fig.colorbar(pixels, ax=ax, extend='max', label='NO2 column (10$^{15}$ molecules cm$^{-2}$)')
plt.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What does ozone look like above the ground? A lidar answers with a curtain: time along the bottom, height up the side, color for ozone. The atmoz package keeps TOLNet lidar data as a dictionary of days, one DataFrame per day, with a row for each profile and a column for each altitude, and <code>plot_curtain</code> draws it. Ours is the NASA JPL SMOL-2 lidar at Hart Miller Island, Maryland, from 28 to 31 July 2025.
</div>

In [ ]:
# OPTIONAL live fetch: TOLNet lidar ozone profiles through atmoz.
# USE_LIVE = False keeps the notebook offline and reads the bundled copy of the same request.
import logging
from atmoz.lidar import TOLNET

logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)   # silences a harmless missing-font warning
tolnet = TOLNET()
tolnet.nrt = False                                                      # no "near real time" stamp on old data

USE_LIVE = False

lidar = None
if USE_LIVE:
    try:
        tolnet.import_data(min_date='2025-07-28', max_date='2025-07-31',
                           product_type=[4], processing_type=[1], instrument_group=[8])
        lidar = tolnet.data[list(tolnet.data)[0]]
        print('Fetched live.')
    except Exception as err:
        print('Live fetch failed (' + type(err).__name__ + '). Falling back to the bundled file.')

if lidar is None:
    profiles = pd.read_parquet(DATA / 'tolnet_smol2_hartmiller_20250728_31.parquet')
    profiles.columns = profiles.columns.astype(float)                   # the altitude labels were saved as text
    lidar = {day: frame for day, frame in profiles.groupby(profiles.index.strftime('%Y-%m-%d'))}
    print('Using the bundled file.')

{day: frame.shape for day, frame in lidar.items()}      # profiles x altitudes, per day

In [ ]:
tolnet.plot_curtain(lidar, title='NASA JPL SMOL-2 ozone lidar, Hart Miller Island, MD')

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        🎯 The vocabulary, at a glance
    </div>
    <ul>
        <li>
            <b>Xarray:</b> a <code>Dataset</code> of labeled arrays. <code>.sel</code> picks by label, <code>.mean('time')</code> collapses an axis, <code>open_dataset</code> reads NetCDF.
        </li>
        <li>
            <b>GeoPandas:</b> a table with a <code>geometry</code> column and a CRS. <code>to_crs</code> changes coordinate system, <code>sjoin</code> joins by location.
        </li>
        <li>
            <b>Cartopy:</b> <code>projection=</code> decides the flat picture, <code>transform=</code> says what the numbers you pass in mean.
        </li>
        <li>
            <b>MetPy:</b> <code>units</code> attach meaning to numbers, <code>metpy.calc</code> does the atmospheric formulas, <code>SkewT</code> draws soundings.
        </li>
        <li>
            <b>Matplotlib:</b> <code>subplot_mosaic</code>, labeled colorbars, annotations and <code>savefig</code> turn a plot into a figure.
        </li>
        <li>
            <b>Plotly:</b> <code>px.scatter</code> and <code>px.line</code> make plots that respond to the mouse.
        </li>
        <li>
            <b>ACT, PyRSIG, atmoz:</b> read ARM files, fetch satellite and monitor data from RSIG, and fetch and draw TOLNet lidar. Each needs <code>pip install</code>.
        </li>
    </ul>
    <b>Bring one real problem:</b> which of these files looks most like the data you work with, and which package would open it?
</div>